# 节点 03 — 反向传播（1986）

**本 notebook 做什么**：
1. 从零用 NumPy 手撕两层神经网络 + 反向传播
2. 解决节点 02 留下的悬念：用反向传播训练网络解决 XOR
3. 可视化损失曲线，直观感受"学习"是怎么发生的

**只需要知道**：Python 基础、numpy 数组运算。不需要微积分。

## 第 0 步：搭好工具箱

In [1]:
import numpy as np
import matplotlib
matplotlib.use('Agg')  # headless rendering
import matplotlib.pyplot as plt

np.random.seed(42)  # 固定随机种子，让结果可复现

# XOR 数据集：这是节点 02 里感知机无法学会的问题
X = np.array([[0,0],[0,1],[1,0],[1,1]], dtype=float)  # 4 个输入
y = np.array([[0],[1],[1],[0]], dtype=float)           # XOR 的正确答案

print('输入 X:')
print(X)
print('目标 y (XOR):')
print(y.flatten())

输入 X:
[[0. 0.]
 [0. 1.]
 [1. 0.]
 [1. 1.]]
目标 y (XOR):
[0. 1. 1. 0.]


## 第 1 步：激活函数 sigmoid

sigmoid 把任意数字压到 0~1 之间。输出可以理解为"概率"或"激活程度"。

$$\sigma(x) = \frac{1}{1 + e^{-x}}$$

它的导数（变化率）有个漂亮的形式：

$$\sigma'(x) = \sigma(x) \cdot (1 - \sigma(x))$$

这在反向传播里会用到。

In [2]:
def sigmoid(x):
    """sigmoid 激活函数"""
    return 1.0 / (1.0 + np.exp(-x))

def sigmoid_deriv(s):
    """sigmoid 的导数，传入 s = sigmoid(x) 的值"""
    return s * (1.0 - s)

# 快速验证：sigmoid(0) 应该是 0.5
print(f'sigmoid(0) = {sigmoid(0):.4f}  (期望: 0.5000)')
print(f'sigmoid(100) = {sigmoid(100):.4f}  (期望: ~1.0000)')
print(f'sigmoid(-100) = {sigmoid(-100):.4f}  (期望: ~0.0000)')

# 验证导数公式
s = sigmoid(2.0)
print(f'sigmoid_deriv(sigmoid(2)) = {sigmoid_deriv(s):.6f}')

sigmoid(0) = 0.5000  (期望: 0.5000)
sigmoid(100) = 1.0000  (期望: ~1.0000)
sigmoid(-100) = 0.0000  (期望: ~0.0000)
sigmoid_deriv(sigmoid(2)) = 0.104994


## 第 2 步：初始化网络权重

网络结构：2 输入 → 4 隐藏节点 → 1 输出

```
x1, x2  →  [隐藏层: 4个节点]  →  [输出层: 1个节点]  →  ŷ
```

每一层都有权重矩阵 W 和偏置向量 b。
权重随机初始化（不能全是 0，否则所有节点学到一样的东西）。

In [3]:
input_size  = 2
hidden_size = 4
output_size = 1
lr = 0.5  # 学习率：每次更新权重的步伐大小

# 第一层权重：形状 (2, 4)，即每个输入连接到 4 个隐藏节点
W1 = np.random.randn(input_size, hidden_size) * 0.5
b1 = np.zeros((1, hidden_size))

# 第二层权重：形状 (4, 1)，4 个隐藏节点连接到 1 个输出
W2 = np.random.randn(hidden_size, output_size) * 0.5
b2 = np.zeros((1, output_size))

print(f'W1 形状: {W1.shape}  (输入尺寸 x 隐藏尺寸)')
print(f'W2 形状: {W2.shape}  (隐藏尺寸 x 输出尺寸)')

W1 形状: (2, 4)  (输入尺寸 x 隐藏尺寸)
W2 形状: (4, 1)  (隐藏尺寸 x 输出尺寸)


## 第 3 步：前向传播

前向传播就是"正常跑一遍网络"，从输入计算出预测值。

1. 隐藏层输入 = X · W1 + b1
2. 隐藏层输出 h = sigmoid(隐藏层输入)
3. 输出层输入 = h · W2 + b2
4. 预测值 ŷ = sigmoid(输出层输入)

In [4]:
def forward(X, W1, b1, W2, b2):
    """前向传播，返回每一层的中间值（反向传播时需要）"""
    z1 = X @ W1 + b1          # 隐藏层线性组合
    h  = sigmoid(z1)           # 隐藏层激活
    z2 = h @ W2 + b2           # 输出层线性组合
    y_hat = sigmoid(z2)        # 最终预测值
    return z1, h, z2, y_hat

# 测试前向传播（初始权重是随机的，所以预测不对，这是正常的）
_, _, _, y_hat = forward(X, W1, b1, W2, b2)
print('训练前的预测值（随机初始权重）:')
for i in range(4):
    print(f'  输入{X[i].astype(int)} -> 预测={y_hat[i,0]:.4f}, 正确答案={int(y[i,0])}')

训练前的预测值（随机初始权重）:
  输入[0 0] -> 预测=0.4467, 正确答案=0
  输入[0 1] -> 预测=0.4303, 正确答案=1
  输入[1 0] -> 预测=0.4270, 正确答案=1
  输入[1 1] -> 预测=0.4126, 正确答案=0


## 第 4 步：反向传播（核心）

反向传播用链式法则，从损失出发，逆向计算每个权重对损失的"贡献"（梯度）。

**损失函数**：均方误差  
`L = mean((y - ŷ)²)`

**反向推导顺序**（从右往左）：
1. `dL/dŷ = -(y - ŷ)` （均方误差对预测值的导数，加了 1/N 平均的系数后化简）
2. `dŷ/dz2 = sigmoid_deriv(ŷ)` （sigmoid 的导数）
3. 链式法则：`dL/dz2 = dL/dŷ * dŷ/dz2`
4. `dL/dW2 = h.T @ dL/dz2`
5. 再往前一层，同理算 `dL/dW1`

In [5]:
def backward(X, y, W2, z1, h, y_hat):
    """反向传播，返回各权重的梯度"""
    n = X.shape[0]  # 样本数量（这里是 4）

    # 输出层梯度
    dL_dy_hat = -(y - y_hat)                      # L 对预测值的梯度
    dL_dz2    = dL_dy_hat * sigmoid_deriv(y_hat)  # 链式法则
    dL_dW2    = h.T @ dL_dz2 / n                  # W2 的梯度
    dL_db2    = dL_dz2.mean(axis=0, keepdims=True)

    # 隐藏层梯度（继续链式法则往前传）
    dL_dh  = dL_dz2 @ W2.T
    dL_dz1 = dL_dh * sigmoid_deriv(h)
    dL_dW1 = X.T @ dL_dz1 / n
    dL_db1 = dL_dz1.mean(axis=0, keepdims=True)

    return dL_dW1, dL_db1, dL_dW2, dL_db2

def loss_mse(y, y_hat):
    return np.mean((y - y_hat) ** 2)

print('反向传播函数定义完毕')

反向传播函数定义完毕


## 第 5 步：训练循环

把前向传播 + 反向传播 + 权重更新重复很多次。
每次叫做一个"epoch"（轮次）。

In [6]:
epochs = 10000
loss_history = []

for epoch in range(epochs):
    # 前向传播
    z1, h, z2, y_hat = forward(X, W1, b1, W2, b2)

    # 计算损失
    L = loss_mse(y, y_hat)
    loss_history.append(L)

    # 反向传播
    dW1, db1, dW2, db2 = backward(X, y, W2, z1, h, y_hat)

    # 更新权重：往梯度的反方向走一小步
    W1 -= lr * dW1
    b1 -= lr * db1
    W2 -= lr * dW2
    b2 -= lr * db2

    if epoch % 2000 == 0:
        print(f'Epoch {epoch:5d}: Loss = {L:.6f}')

print(f'\n训练完成！最终 Loss = {loss_history[-1]:.6f}')

Epoch     0: Loss = 0.255675
Epoch  2000: Loss = 0.249659
Epoch  4000: Loss = 0.202610
Epoch  6000: Loss = 0.022670
Epoch  8000: Loss = 0.005194



训练完成！最终 Loss = 0.002661


## 第 6 步：验证——XOR 解决了吗？

In [7]:
_, _, _, final_pred = forward(X, W1, b1, W2, b2)

print('训练后的预测结果：')
all_correct = True
for i in range(4):
    pred_class = int(final_pred[i, 0] > 0.5)
    correct = pred_class == int(y[i, 0])
    if not correct:
        all_correct = False
    status = 'OK' if correct else 'WRONG'
    print(f'  输入{X[i].astype(int)} -> 预测={final_pred[i,0]:.4f} -> 类别={pred_class} (正确={int(y[i,0])}) [{status}]')

accuracy = sum(int(final_pred[i,0] > 0.5) == int(y[i,0]) for i in range(4)) / 4
print(f'\n准确率: {accuracy:.2%}')
print(f'XOR 已解决: {all_correct}')

# 硬性断言——这是 notebook 必须通过的检验
assert accuracy == 1.0, f'准确率 {accuracy} != 1.0，训练未成功'

训练后的预测结果：
  输入[0 0] -> 预测=0.0562 -> 类别=0 (正确=0) [OK]
  输入[0 1] -> 预测=0.9507 -> 类别=1 (正确=1) [OK]
  输入[1 0] -> 预测=0.9506 -> 类别=1 (正确=1) [OK]
  输入[1 1] -> 预测=0.0512 -> 类别=0 (正确=0) [OK]

准确率: 100.00%
XOR 已解决: True


## 第 7 步：可视化损失曲线

In [8]:
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(loss_history, color='steelblue', linewidth=1.5)
ax.set_xlabel('训练轮次 (Epoch)')
ax.set_ylabel('损失 (MSE)')
ax.set_title('反向传播训练 XOR 的损失曲线')
ax.set_yscale('log')  # 对数坐标更容易看出收敛过程
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('../figures/03-loss-curve.png', dpi=80)
plt.close()
print('损失曲线已保存到 figures/03-loss-curve.png')

损失曲线已保存到 figures/03-loss-curve.png


/var/folders/f7/ly4gmfkx0_j9szgtny55xxd80000gn/T/ipykernel_13947/3032871901.py:8: UserWarning: Glyph 35757 (\N{CJK UNIFIED IDEOGRAPH-8BAD}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
/var/folders/f7/ly4gmfkx0_j9szgtny55xxd80000gn/T/ipykernel_13947/3032871901.py:8: UserWarning: Glyph 32451 (\N{CJK UNIFIED IDEOGRAPH-7EC3}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
/var/folders/f7/ly4gmfkx0_j9szgtny55xxd80000gn/T/ipykernel_13947/3032871901.py:8: UserWarning: Glyph 36718 (\N{CJK UNIFIED IDEOGRAPH-8F6E}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
/var/folders/f7/ly4gmfkx0_j9szgtny55xxd80000gn/T/ipykernel_13947/3032871901.py:8: UserWarning: Glyph 27425 (\N{CJK UNIFIED IDEOGRAPH-6B21}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
/var/folders/f7/ly4gmfkx0_j9szgtny55xxd80000gn/T/ipykernel_13947/3032871901.py:8: UserWarning: Glyph 25439 (\N{CJK UNIFIED IDEOGRAPH-635F}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
/var/folders/f7/ly4gmfkx0

## 第 8 步：直觉验证——隐藏层在干什么？

反向传播的核心贡献：隐藏层会**自动**学到对问题有用的内部表示。
让我们看看经过训练后，隐藏层把 XOR 的 4 个输入映射到了什么位置。

In [9]:
# 计算每个输入在隐藏层的激活值
_, h_final, _, _ = forward(X, W1, b1, W2, b2)

print('隐藏层学到的表示（4x4 矩阵，每行一个输入的隐藏层输出）：')
labels = ['(0,0)->0', '(0,1)->1', '(1,0)->1', '(1,1)->0']
for i, label in enumerate(labels):
    h_vals = '  '.join(f'{v:.3f}' for v in h_final[i])
    print(f'  {label}: [{h_vals}]')

print()
print('关键直觉：两个输出为 1 的输入（(0,1) 和 (1,0)）在隐藏层里')
print('映射到了相近的位置，而两个输出为 0 的输入映射到了不同的位置。')
print('这就是"学到了内部表示"的意思。')

隐藏层学到的表示（4x4 矩阵，每行一个输入的隐藏层输出）：
  (0,0)->0: [0.021  0.517  0.015  0.068]
  (0,1)->1: [0.187  0.480  0.192  0.968]
  (1,0)->1: [0.211  0.499  0.169  0.969]
  (1,1)->0: [0.738  0.462  0.760  1.000]

关键直觉：两个输出为 1 的输入（(0,1) 和 (1,0)）在隐藏层里
映射到了相近的位置，而两个输出为 0 的输入映射到了不同的位置。
这就是"学到了内部表示"的意思。


## 第 9 步：与节点 02 的对比

在节点 02 里，感知机（单层线性模型）对 XOR 永远卡在约 50% 准确率。  
两层网络 + 反向传播，同样的数据，10000 轮后达到 100%。

**这就是 1986 年那篇 Nature 论文的意义。**

In [10]:
# 模拟节点 02 里感知机的行为：在 XOR 上最高准确率约 75%（只能对 3/4）
# 这里简单展示：用线性模型（没有隐藏层）尝试学 XOR
W_linear = np.random.randn(2, 1) * 0.5
b_linear = np.zeros((1, 1))
lr_linear = 0.1

for _ in range(10000):
    pred_linear = sigmoid(X @ W_linear + b_linear)
    dW = -(y - pred_linear) * sigmoid_deriv(pred_linear)
    W_linear -= lr_linear * (X.T @ dW / 4)
    b_linear -= lr_linear * dW.mean()

pred_linear_final = sigmoid(X @ W_linear + b_linear)
acc_linear = sum(int(pred_linear_final[i,0] > 0.5) == int(y[i,0]) for i in range(4)) / 4

print(f'单层线性网络（无隐藏层）在 XOR 上的准确率：{acc_linear:.2%}')
print(f'两层网络（有隐藏层）在 XOR 上的准确率：{accuracy:.2%}')
print()
print('结论：单层网络最多对 3/4（75%），两层网络可以做到 100%。')

# 确认单层网络无法完美解决 XOR
assert acc_linear < 1.0, '预期单层网络无法 100% 解决 XOR'

单层线性网络（无隐藏层）在 XOR 上的准确率：75.00%
两层网络（有隐藏层）在 XOR 上的准确率：100.00%

结论：单层网络最多对 3/4（75%），两层网络可以做到 100%。


## 总结

在这个 notebook 里，你从零实现了：

1. **Sigmoid 激活函数**及其导数
2. **两层神经网络**的前向传播
3. **反向传播**：用链式法则把误差传回每一层
4. **梯度下降**更新权重
5. 解决了节点 02 留下的 XOR 悬题

这正是 Rumelhart、Hinton 和 Williams 在 1986 年 Nature 论文里展示的核心算法。

---

**下一节**：[节点 04 — 卷积网络 LeNet（1989/1998）](../docs/04-lenet-1989.md) *(待建)*